# Battery fleet investigator (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<your-username>/battery-fit-agent/blob/main/notebooks/colab_investigator.ipynb)

Runs the **investigator mode** of `battery-fit-agent` on all 139 Severson fast-charge cells:

1. **Fit** seven degradation laws to every cell (deterministic tools, no LLM).
2. **Triage** each cell against the fleet (misfit and knee z-scores, shape vs. the fleet's law, data quality;
   tests that never got near end of life are excluded from the knee comparison).
3. **Investigate** each high-priority cell with a tool-using Gemini agent. It decides which tools to call
   (overview, end-of-life check, outlier locations, fleet comparison, windowed refits), sees the diagnostic plot,
   and returns a structured verdict. A deterministic verifier checks the verdict against the tool results and
   sends it back once if they disagree.
4. **Report**, written from the verdicts only, then scored against hand labels and a rule-based baseline.

Needs a free Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey), stored as a Colab secret
named `GOOGLE_API_KEY` (key icon in the left sidebar). No GPU needed.

In [ ]:
!git clone -q https://github.com/<your-username>/battery-fit-agent.git
%cd battery-fit-agent
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

MODEL = "gemini-2.5-flash"     # any Gemini model your key can use, e.g. a newer Flash or a Pro model
RPM = 8                        # requests per minute; keep under your tier's limit

## The agent: outer graph and investigator subgraph

In [ ]:
from IPython.display import Markdown, display
from battery_fit_agent import Config
from battery_fit_agent.llm import get_chat_model
from battery_fit_agent.graph_v2 import build_fleet_agent

cfg = Config(out_dir="agent_outputs_investigator"); cfg.make_dirs()
llm = get_chat_model("gemini", MODEL, requests_per_minute=RPM)
agent = build_fleet_agent(cfg, llm, which="high", send_plot=True)
display(Markdown("```mermaid\n" + agent.get_graph().draw_mermaid() + "\n```"))

## Run

Fitting all 139 cells takes a few minutes; the investigation then makes roughly 6–8 model calls per cell.
Use `which="all"` to investigate every flagged cell (about 3× more calls), or pass `cells=[...]` with `which="cells"`.

In [ ]:
final = agent.invoke({"queue": [], "verdicts": [], "report": "", "shortlist": []}, {"recursion_limit": 500})
import pandas as pd
verdicts = pd.DataFrame(final["verdicts"])
verdicts[["cell_id", "category", "confidence", "tool_calls", "revised", "baseline", "triage_reasons"]]

## Report (written from the verdicts only)

In [ ]:
display(Markdown(final["report"]))

## Agreement with hand labels

`results/manual_labels.csv` holds labels made by inspecting the plots of the ten high-priority cells from the
first run. The rule baseline is a fixed decision list over the same facts the agent can query.

In [ ]:
from battery_fit_agent.evaluate import evaluate
score, table = evaluate(verdicts, "results/manual_labels.csv")
print(score)
table

## One investigation, step by step

In [ ]:
cell = verdicts.iloc[0]["cell_id"]
display(Markdown(open(f"{cfg.out_dir}/traces/{cell}.md").read()))

## Plots of the investigated cells

In [ ]:
from IPython.display import Image
for _, v in verdicts.iterrows():
    print(f"{v.cell_id}: {v.category} ({v.confidence})")
    display(Image(f"{cfg.plot_dir}/{v.cell_id}.png", width=600))